# 16 — Automated Excel Report

Turns a scored dataset into a formatted, management-ready Excel workbook:
a KPI summary sheet, segment breakdowns with an embedded chart, a
conditionally-formatted top-risk list, and the underlying data.

By default it reads the predictions file produced by notebook 04 (or 08/12);
point it at any scored CSV of yours.

**OUTPUT**: `outputs/portfolio_report.xlsx`

In [1]:
# ============================================================
# SETUP — run this cell first (no edits needed)
# ============================================================
# If any import fails, run in a notebook cell:
#   %pip install pandas numpy matplotlib seaborn scikit-learn sqlalchemy joblib openpyxl
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# All files this notebook produces are saved here:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Setup complete. Outputs will be saved to:", OUTPUT_DIR)

Setup complete. Outputs will be saved to: outputs


In [2]:
# ============================================================
# SAMPLE DATA GENERATOR (used only when DATA_SOURCE = "sample")
# ============================================================
# Creates a synthetic consumer-lending dataset so every cell below runs
# end-to-end even before you plug in your own data. Just run this cell.
def make_sample_lending_data(n=5000, seed=42):
    rng = np.random.default_rng(seed)
    fico = rng.normal(690, 55, n).clip(500, 850).round()
    dti = rng.normal(28, 10, n).clip(1, 65).round(1)
    loan_amount = rng.lognormal(9.4, 0.5, n).clip(1000, 50000).round(-2)
    income = rng.lognormal(11.1, 0.45, n).clip(15000, 400000).round(-2)
    utilization = rng.beta(2, 3, n).round(3) * 100
    tenure_months = rng.integers(0, 240, n)
    grade = rng.choice(list("ABCDE"), n, p=[0.25, 0.30, 0.25, 0.13, 0.07])
    purpose = rng.choice(["debt_consolidation", "credit_card", "home_improvement",
                          "auto", "medical", "other"], n,
                         p=[0.38, 0.22, 0.13, 0.12, 0.06, 0.09])
    state = rng.choice(["CA", "TX", "NY", "FL", "IL", "PA", "OH", "GA"], n)
    grade_rate = pd.Series(grade).map({"A": 7.5, "B": 10.5, "C": 13.5, "D": 17.0, "E": 21.0}).values
    interest_rate = (grade_rate - 0.010 * (fico - 690) + 0.02 * (dti - 28)
                     + rng.normal(0, 0.8, n)).clip(5, 30).round(2)
    origination_date = (pd.Timestamp("2023-01-01")
                        + pd.to_timedelta(rng.integers(0, 36, n) * 30, unit="D")).normalize()
    logit = (-4.2
             - 0.012 * (fico - 690)
             + 0.045 * (dti - 28)
             + 0.018 * (utilization - 40)
             + 0.35 * np.isin(grade, ["D", "E"]).astype(float)
             + 0.20 * (purpose == "debt_consolidation").astype(float))
    p_default = 1 / (1 + np.exp(-logit))
    default_flag = rng.binomial(1, p_default)
    df = pd.DataFrame({
        "loan_id": np.arange(1, n + 1),
        "origination_date": origination_date,
        "fico_score": fico, "dti": dti, "loan_amount": loan_amount,
        "annual_income": income, "revolving_utilization": utilization,
        "employment_tenure_months": tenure_months, "grade": grade,
        "loan_purpose": purpose, "state": state,
        "interest_rate": interest_rate, "default_flag": default_flag,
    })
    for col, frac in [("dti", 0.03), ("annual_income", 0.05), ("employment_tenure_months", 0.02)]:
        df.loc[df.sample(frac=frac, random_state=seed).index, col] = np.nan
    return df

print("Sample data generator defined.")

Sample data generator defined.


In [3]:
# ============================================================
# INPUT — EDIT THIS CELL
# ============================================================
SCORED_CSV = "outputs/xgboost_predictions.csv"  # <-- any scored file
PROB_COL = "xgboost_probability"                # <-- probability/score column
TARGET_COL = "default_flag"                     # actual outcome (or None)
SEGMENT_COLS = ["grade", "loan_purpose"]        # categorical columns to break down by
AMOUNT_COL = "loan_amount"                      # exposure column (or None)
REPORT_TITLE = "Portfolio Risk Report"

if Path(SCORED_CSV).exists():
    rpt = pd.read_csv(SCORED_CSV)
else:
    # Fallback so the notebook runs standalone: quick model on sample data
    print(f"{SCORED_CSV} not found — building sample scored data instead.")
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    rpt = make_sample_lending_data()
    Xr = rpt.drop(columns=["loan_id", "default_flag", "origination_date"])
    numf = Xr.select_dtypes(include=np.number).columns.tolist()
    catf = Xr.select_dtypes(include=["object", "string"]).columns.tolist()
    quick = Pipeline([("p", ColumnTransformer([
        ("n", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), numf),
        ("c", OneHotEncoder(handle_unknown="ignore"), catf)])),
        ("m", LogisticRegression(max_iter=2000))]).fit(Xr[numf + catf], rpt["default_flag"])
    PROB_COL = "probability"
    rpt[PROB_COL] = quick.predict_proba(Xr[numf + catf])[:, 1]

print(f"Report input: {len(rpt):,} rows | probability column: {PROB_COL}")

Report input: 5,000 rows | probability column: xgboost_probability


In [4]:
# ============================================================
# Build the report tables
# ============================================================
kpis = {
    "Report generated": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "Records": f"{len(rpt):,}",
    "Average predicted risk": f"{rpt[PROB_COL].mean():.2%}",
    "High risk (prob > 20%)": f"{(rpt[PROB_COL] > 0.20).mean():.2%} of records",
}
if TARGET_COL and TARGET_COL in rpt.columns:
    kpis["Actual bad rate"] = f"{rpt[TARGET_COL].mean():.2%}"
if AMOUNT_COL and AMOUNT_COL in rpt.columns:
    amt = pd.to_numeric(rpt[AMOUNT_COL], errors="coerce")
    kpis["Total exposure"] = f"${amt.sum():,.0f}"
    kpis["Expected loss (prob x amount)"] = f"${(rpt[PROB_COL] * amt).sum():,.0f}"

segments = {}
for c in SEGMENT_COLS:
    if c not in rpt.columns:
        continue
    agg = {"records": (PROB_COL, "size"), "avg_predicted_risk": (PROB_COL, "mean")}
    if TARGET_COL and TARGET_COL in rpt.columns:
        agg["actual_bad_rate"] = (TARGET_COL, "mean")
    if AMOUNT_COL and AMOUNT_COL in rpt.columns:
        agg["total_amount"] = (AMOUNT_COL, "sum")
    segments[c] = rpt.groupby(c).agg(**agg).round(4).sort_values("avg_predicted_risk",
                                                                 ascending=False)
top_risk = rpt.sort_values(PROB_COL, ascending=False).head(50)

for k, v in kpis.items():
    print(f"{k:32s}: {v}")

Report generated                : 2026-07-09 15:29
Records                         : 5,000
Average predicted risk          : 31.13%
High risk (prob > 20%)          : 64.28% of records
Actual bad rate                 : 2.72%
Total exposure                  : $68,188,700
Expected loss (prob x amount)   : $21,263,193


In [5]:
# ============================================================
# OUTPUT — write the formatted workbook
# ============================================================
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.chart import BarChart, Reference
from openpyxl.utils import get_column_letter

report_path = OUTPUT_DIR / "portfolio_report.xlsx"
HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
HEADER_FONT = Font(color="FFFFFF", bold=True)

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    # --- Summary sheet ---
    pd.DataFrame(list(kpis.items()), columns=["Metric", "Value"]) \
      .to_excel(writer, sheet_name="Summary", index=False, startrow=2)
    ws = writer.sheets["Summary"]
    ws["A1"] = REPORT_TITLE
    ws["A1"].font = Font(size=16, bold=True, color="1F4E79")
    for c in ("A3", "B3"):
        ws[c].fill, ws[c].font = HEADER_FILL, HEADER_FONT
    ws.column_dimensions["A"].width = 34
    ws.column_dimensions["B"].width = 26

    # --- One sheet per segment, with an embedded risk chart ---
    for seg, table in segments.items():
        sheet = f"By {seg}"[:31]
        table.to_excel(writer, sheet_name=sheet)
        ws = writer.sheets[sheet]
        for col in range(1, table.shape[1] + 2):
            cell = ws.cell(row=1, column=col)
            cell.fill, cell.font = HEADER_FILL, HEADER_FONT
            ws.column_dimensions[get_column_letter(col)].width = 20
        risk_col = list(table.columns).index("avg_predicted_risk") + 2
        chart = BarChart()
        chart.title = f"Avg predicted risk by {seg}"
        chart.add_data(Reference(ws, min_col=risk_col, min_row=1,
                                 max_row=len(table) + 1), titles_from_data=True)
        chart.set_categories(Reference(ws, min_col=1, min_row=2, max_row=len(table) + 1))
        chart.height, chart.width = 8, 16
        ws.add_chart(chart, f"{get_column_letter(table.shape[1] + 3)}2")

    # --- Top-risk records with a red->green color scale on probability ---
    top_risk.to_excel(writer, sheet_name="Top 50 risk", index=False)
    ws = writer.sheets["Top 50 risk"]
    for col in range(1, top_risk.shape[1] + 1):
        cell = ws.cell(row=1, column=col)
        cell.fill, cell.font = HEADER_FILL, HEADER_FONT
    pcol = get_column_letter(list(top_risk.columns).index(PROB_COL) + 1)
    ws.conditional_formatting.add(
        f"{pcol}2:{pcol}{len(top_risk) + 1}",
        ColorScaleRule(start_type="min", start_color="63BE7B",
                       end_type="max", end_color="F8696B"))

    # --- Raw data ---
    rpt.head(10000).to_excel(writer, sheet_name="Data", index=False)

print(f"Saved {report_path}")
print("Sheets: Summary | " + " | ".join(f"By {s}" for s in segments) + " | Top 50 risk | Data")

Saved outputs/portfolio_report.xlsx
Sheets: Summary | By grade | By loan_purpose | Top 50 risk | Data
